In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import norm, skewnorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from scipy.stats import rankdata
import warnings, os, time

warnings.filterwarnings("ignore")

MODEL_NAME = "MAQR"
DATASET_NAME = "Synthetic_Hetero"
RESULTS_DIR = "results"
PLOTS_DIR = "plots"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SPLIT_SEED = 58
SEEDS = [42, 58, 123, 256, 789]
N_RUNS = len(SEEDS)
DEVICE = torch.device("cpu")

QUANTILES = [
    0.01, 0.025, 0.03, 0.05, 0.09, 0.10, 0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90, 0.91, 0.93, 0.95, 0.97, 0.975, 0.99,
]
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
CONTAMINATION_LEVELS = [0.02, 0.05, 0.10]
EP = 100; H = 100; LR = 0.01; BS = 32

plt.rcParams.update({
    "figure.figsize": (12, 5), "font.size": 11, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
})
C_PRED = "#2563EB"; C_TRUE = "#DC2626"; C_CLEAN = "#3B82F6"
C_CONTAM = "#F59E0B"; C_MODEL = "#7C3AED"; C_BAR = "#0EA5E9"

print(f"Config: {MODEL_NAME} | {DATASET_NAME} | {N_RUNS} seeds | Device: {DEVICE}")


Config: MAQR | Synthetic_Hetero | 5 seeds | Device: cpu


In [2]:
np.random.seed(SPLIT_SEED)
N_SAMPLES = 1000
X_raw = np.random.uniform(0, 2, N_SAMPLES)
SIGMA_SCALE = 1.0

def true_mean(x):
    return x

def true_quantile(x, tau):
    return true_mean(x) + (SIGMA_SCALE * x) * norm.ppf(tau)

Y_clean = true_mean(X_raw) + np.random.normal(0, SIGMA_SCALE * X_raw, N_SAMPLES)
X_col = X_raw.reshape(-1, 1)
DIM = 1

Xtv, X_test, ytv, y_test = train_test_split(X_col, Y_clean, test_size=0.20, random_state=SPLIT_SEED)
X_train, X_val, y_train_clean, y_val = train_test_split(Xtv, ytv, test_size=0.20, random_state=SPLIT_SEED)
print(f"Split: Train={X_train.shape[0]} Val={X_val.shape[0]} Test={X_test.shape[0]}")


Split: Train=640 Val=160 Test=200


In [3]:
def inject_outliers(y, frac, method, seed=None):
    rng = np.random.RandomState(seed)
    y = y.copy()
    n = len(y)
    n_out = max(1, int(frac * n))
    idx = rng.choice(n, n_out, replace=False)
    mu, sig = y.mean(), y.std()
    if method == "gaussian":
        y[idx] = rng.normal(mu, np.sqrt(5) * sig, n_out)
    elif method == "multiply":
        y[idx] = y[idx] * rng.choice([-1, 1], n_out) * 10
    elif method == "uniform":
        y[idx] = rng.uniform(y.min() - 3 * sig, y.max() + 3 * sig, n_out)
    elif method == "skewed":
        y[idx] = skewnorm.rvs(a=10, loc=mu, scale=3 * sig, size=n_out, random_state=rng)
    return y

cont_sets = {}
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        cont_sets[(ot, cl)] = inject_outliers(y_train_clean, cl, ot, seed=SPLIT_SEED)
print(f"{len(cont_sets)} contaminated training sets created")


12 contaminated training sets created


In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train.ravel(), y_train_clean, s=10, alpha=0.5, color=C_CLEAN, label="Clean Data")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title(f"{DATASET_NAME} — Clean Training Data")
ax.legend(frameon=False)
plt.savefig(f"{PLOTS_DIR}/scatter_clean.png"); plt.close()

for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].scatter(X_train.ravel(), y_train_clean, s=10, alpha=0.5, color=C_CLEAN)
        axes[0].set_title("Clean Data"); axes[0].set_xlabel("X"); axes[0].set_ylabel("Y")
        axes[1].scatter(X_train.ravel(), cont_sets[(ot, cl)], s=10, alpha=0.5, color=C_CONTAM)
        axes[1].set_title(f"{ot.capitalize()} {int(cl*100)}%"); axes[1].set_xlabel("X"); axes[1].set_ylabel("Y")
        for ax in axes:
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(f"{PLOTS_DIR}/scatter_{ot}_{int(cl*100)}pct.png"); plt.close()
print(f"Scatter plots saved")


Scatter plots saved


In [5]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, H), nn.ReLU(), nn.Linear(H, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()

def train_model(X, y, loss_fn, epochs=EP):
    model = MLP(DIM).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=LR)
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
    model.eval()
    return model

def train_mse(X, y, epochs=EP):
    return train_model(X, y, lambda p, t: nn.functional.mse_loss(p, t), epochs)

def pinball_loss(pred, target, tau):
    e = target - pred
    return torch.mean(torch.max(tau * e, (tau - 1) * e))

class MAQR:
    def __init__(self, base_fn, k=50, max_anchors=2000):
        self.base_fn = base_fn
        self.k = k
        self.max_anchors = max_anchors

    def fit(self, X, y):
        self.residuals = y.ravel() - self.base_fn(X).ravel()
        n = len(X)
        if n > self.max_anchors:
            idx = np.random.choice(n, self.max_anchors, replace=False)
            Xs, rs = X[idx], self.residuals[idx]
        else:
            Xs, rs = X, self.residuals
        self.nn = NearestNeighbors(n_neighbors=min(self.k, len(Xs)))
        self.nn.fit(Xs)
        Dx, Dp, Dr = [], [], []
        for i in range(len(Xs)):
            _, idx = self.nn.kneighbors(Xs[i:i+1])
            nr = rs[idx[0]]
            pv = rankdata(nr, "ordinal") / (len(nr) + 1)
            for j in range(len(idx[0])):
                Dx.append(Xs[i]); Dp.append(pv[j]); Dr.append(nr[j])
        self.lm = LinearRegression()
        self.lm.fit(np.hstack([np.array(Dx), np.array(Dp).reshape(-1, 1)]), np.array(Dr))

    def predict(self, X, quantiles):
        bp = self.base_fn(X).ravel()
        R = {}
        for tau in quantiles:
            R[tau] = bp + self.lm.predict(np.hstack([X, np.full((len(X), 1), tau)]))
        return R

print("MAQR model defined")


MAQR model defined


In [6]:
def compute_shift_rmse(preds_clean, preds_contam, quantiles):
    results = {}
    for tau in quantiles:
        diff = preds_clean[tau] - preds_contam[tau]
        results[tau] = np.sqrt(np.mean(diff ** 2))
    return results

def compute_ece(y_test, preds, quantiles):
    results = {}
    for tau in quantiles:
        coverage = np.mean(y_test <= preds[tau])
        results[tau] = abs(coverage - tau)
    return results

def compute_true_quantile_rmse(preds, x_test, quantiles):
    results = {}
    for tau in quantiles:
        tq = true_quantile(x_test.ravel(), tau)
        results[tau] = np.sqrt(mean_squared_error(tq, preds[tau]))
    return results
print("Evaluation functions defined")


Evaluation functions defined


In [7]:
all_preds = {}; all_shift_rmse = {}; all_ece_clean = {}; all_ece_contam = {}; all_true_q_rmse = {}

conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]
print(f"Conditions: {len(conditions)} | Seeds: {N_RUNS}")
t0 = time.time()

for si, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSeed {seed} ({si+1}/{N_RUNS})\n{'='*60}")
    all_preds[seed] = {}; all_shift_rmse[seed] = {}; all_ece_contam[seed] = {}

    for ci, cond in enumerate(conditions):
        cond_label = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        y_tr = y_train_clean if cond == "clean" else cont_sets[cond]

        set_seed(seed)
        mse_model = train_mse(X_train, y_tr)
        base_fn = lambda X, m=mse_model: predict(m, X)

        set_seed(seed)
        maqr = MAQR(base_fn, k=50)
        maqr.fit(X_train, y_tr)
        preds = maqr.predict(X_test, QUANTILES)

        all_preds[seed][cond] = preds
        if cond == "clean":
            all_ece_clean[seed] = compute_ece(y_test, preds, QUANTILES)
            all_true_q_rmse[seed] = compute_true_quantile_rmse(preds, X_test, QUANTILES)
        else:
            all_shift_rmse[seed][cond] = compute_shift_rmse(all_preds[seed]["clean"], preds, QUANTILES)
            all_ece_contam[seed][cond] = compute_ece(y_test, preds, QUANTILES)

        elapsed = time.time() - t0
        print(f"  [{(ci+1)/len(conditions)*100:5.1f}%] {cond_label:25s} | {elapsed/60:.1f}min")

print(f"\nDone: {(time.time()-t0)/60:.1f} min")


Conditions: 13 | Seeds: 5

Seed 42 (1/5)


  [  7.7%] clean                     | 0.0min


  [ 15.4%] gaussian_2pct             | 0.1min


  [ 23.1%] gaussian_5pct             | 0.1min


  [ 30.8%] gaussian_10pct            | 0.1min


...

  [ 69.2%] uniform_5pct              | 1.5min


  [ 76.9%] uniform_10pct             | 1.5min


  [ 84.6%] skewed_2pct               | 1.5min


  [ 92.3%] skewed_5pct               | 1.5min


  [100.0%] skewed_10pct              | 1.6min

Done: 1.6 min


In [8]:
from openpyxl import Workbook
wb = Workbook()
ws_summary = wb.active
ws_summary.title = "Summary"
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ShiftRMSE_seed_{seed}")
    header = ["Condition"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_shift_rmse[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_clean_seed_{seed}")
    ws.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
    vals = all_ece_clean[seed]
    row = ["ECE_clean"] + [round(vals[t], 6) for t in QUANTILES]
    row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
    ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_contam_seed_{seed}")
    ws.append(["Condition"] + [str(t) for t in QUANTILES] + ["Avg"])
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_ece_contam[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"TrueQ_RMSE_seed_{seed}")
    ws.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
    vals = all_true_q_rmse[seed]
    row = ["TrueQ_RMSE"] + [round(vals[t], 6) for t in QUANTILES]
    row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
    ws.append(row)

# Summary
ws_summary.append(["=== Shift-RMSE (mean across seeds) ==="])
ws_summary.append(["Condition"] + [str(t) for t in QUANTILES] + ["Avg"])
for cond in contam_conditions:
    label = f"{cond[0]}_{int(cond[1]*100)}pct"
    means = [np.mean([all_shift_rmse[s][cond][t] for s in SEEDS]) for t in QUANTILES]
    ws_summary.append([label] + [round(m, 6) for m in means] + [round(np.mean(means), 6)])

ws_summary.append([])
ws_summary.append(["=== ECE Clean (mean across seeds) ==="])
ws_summary.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
means_ece = [np.mean([all_ece_clean[s][t] for s in SEEDS]) for t in QUANTILES]
ws_summary.append(["ECE_clean"] + [round(m, 6) for m in means_ece] + [round(np.mean(means_ece), 6)])

ws_summary.append([])
ws_summary.append(["=== True Quantile RMSE (mean across seeds) ==="])
ws_summary.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
means_tq = [np.mean([all_true_q_rmse[s][t] for s in SEEDS]) for t in QUANTILES]
ws_summary.append(["TrueQ_RMSE"] + [round(m, 6) for m in means_tq] + [round(np.mean(means_tq), 6)])

# Alpha sheet
ws_alpha = wb.create_sheet(title="Alpha")
ws_alpha.append(["Method", "Alpha"])
ws_alpha.append([MODEL_NAME, "N/A (no trimming parameter)"])
ws_alpha.append(["k_neighbors", 50])
ws_alpha.append(["second_stage", "LinearRegression"])

xlsx_path = f"{RESULTS_DIR}/{DATASET_NAME}_{MODEL_NAME}_results.xlsx"
wb.save(xlsx_path)
print(f"Saved: {xlsx_path}")


Saved: results/Synthetic_Hetero_MAQR_results.xlsx


In [9]:
sort_idx = np.argsort(X_test.ravel())
X_sorted = X_test.ravel()[sort_idx]
tq_lo = true_quantile(X_sorted, 0.025)
tq_hi = true_quantile(X_sorted, 0.975)

def plot_quantile_curves(preds, y_train_plot, title_suffix, fname):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(X_train.ravel(), y_train_plot, s=8, alpha=0.3, color=C_CONTAM, label="Training Data", zorder=1)
    ax.plot(X_sorted, tq_lo, "-", color=C_TRUE, lw=1.5, label="True Q(0.025)", zorder=2)
    ax.plot(X_sorted, preds[0.025][sort_idx], "--", color=C_PRED, lw=1.5, label="Pred Q(0.025)", zorder=3)
    ax.plot(X_sorted, tq_hi, "-", color=C_TRUE, lw=1.5, label="True Q(0.975)", zorder=2)
    ax.plot(X_sorted, preds[0.975][sort_idx], "--", color=C_PRED, lw=1.5, label="Pred Q(0.975)", zorder=3)
    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.set_title(f"{MODEL_NAME} — {title_suffix}", fontsize=12)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/{fname}"); plt.close()

conditions_plot = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]
for cond in conditions_plot:
    cond_label = "Clean" if cond == "clean" else f"{cond[0].capitalize()} {int(cond[1]*100)}%"
    y_tp = y_train_clean if cond == "clean" else cont_sets[cond]
    fname_tag = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
    for seed in SEEDS:
        plot_quantile_curves(all_preds[seed][cond], y_tp, f"{cond_label} (seed={seed})", f"qcurve_{fname_tag}_seed{seed}.png")
    avg_preds = {tau: np.mean([all_preds[s][cond][tau] for s in SEEDS], axis=0) for tau in QUANTILES}
    plot_quantile_curves(avg_preds, y_tp, f"{cond_label} (avg {N_RUNS} seeds)", f"qcurve_{fname_tag}_avg.png")
print("Quantile curve plots saved")


Quantile curve plots saved


In [10]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in contam_conditions:
    cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
    fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
    per_seed = np.array([[all_shift_rmse[s][cond][t] for t in QUANTILES] for s in SEEDS])
    means, stds = per_seed.mean(0), per_seed.std(0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color=C_BAR, lw=2, markersize=5, capsize=3)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
    ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_avg.png"); plt.close()

    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = [all_shift_rmse[seed][cond][t] for t in QUANTILES]
        ax.plot(x_ticks, vals, "o-", color=C_BAR, lw=2, markersize=5)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
        ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_seed{seed}.png"); plt.close()
print("Shift-RMSE plots saved")


Shift-RMSE plots saved


In [11]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
per_seed = np.array([[all_true_q_rmse[s][t] for t in QUANTILES] for s in SEEDS])
means, stds = per_seed.mean(0), per_seed.std(0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color=C_MODEL, lw=2, markersize=5, capsize=3)
ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Quantile τ"); ax.set_ylabel("RMSE vs True Quantile")
ax.set_title(f"{DATASET_NAME} — True Quantile RMSE (Clean): {MODEL_NAME}")
plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/true_q_rmse_clean_avg.png"); plt.close()

for seed in SEEDS:
    fig, ax = plt.subplots(figsize=(14, 5))
    vals = [all_true_q_rmse[seed][t] for t in QUANTILES]
    ax.plot(x_ticks, vals, "o-", color=C_MODEL, lw=2, markersize=5)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("RMSE vs True Quantile")
    ax.set_title(f"{DATASET_NAME} — True Quantile RMSE (Clean): {MODEL_NAME} (seed={seed})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/true_q_rmse_clean_seed{seed}.png"); plt.close()
print("True quantile RMSE plots saved")


True quantile RMSE plots saved


In [12]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
all_ece_conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in all_ece_conditions:
    if cond == "clean":
        cond_label, fname_tag = "Clean", "clean"
        per_seed = np.array([[all_ece_clean[s][t] for t in QUANTILES] for s in SEEDS])
    else:
        cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
        fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
        per_seed = np.array([[all_ece_contam[s][cond][t] for t in QUANTILES] for s in SEEDS])
    means, stds = per_seed.mean(0), per_seed.std(0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color="#10B981", lw=2, markersize=5, capsize=3)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
    ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_avg.png"); plt.close()

    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = all_ece_clean[seed] if cond == "clean" else all_ece_contam[seed][cond]
        ax.plot(x_ticks, [vals[t] for t in QUANTILES], "o-", color="#10B981", lw=2, markersize=5)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
        ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_seed{seed}.png"); plt.close()
print("ECE plots saved")


ECE plots saved


In [13]:
print(f"{'='*60}")
print(f"{MODEL_NAME} on {DATASET_NAME} — Complete")
print(f"{'='*60}")
print(f"Results: {RESULTS_DIR}/"); print(f"Plots: {PLOTS_DIR}/")

print(f"\n--- Avg True Quantile RMSE (clean) ---")
for tau in [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]:
    vals = [all_true_q_rmse[s][tau] for s in SEEDS]
    print(f"  τ={tau:.3f}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print(f"\n--- Avg Shift-RMSE (multiply 10%) ---")
for tau in [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]:
    vals = [all_shift_rmse[s][("multiply", 0.10)][tau] for s in SEEDS]
    print(f"  τ={tau:.3f}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")


MAQR on Synthetic_Hetero — Complete
Results: results/
Plots: plots/

--- Avg True Quantile RMSE (clean) ---
  τ=0.025: 1.2293 ± 0.0013
  τ=0.050: 0.9939 ± 0.0013
  τ=0.100: 0.7707 ± 0.0012
  τ=0.500: 0.1076 ± 0.0031
  τ=0.900: 0.7695 ± 0.0008
  τ=0.950: 1.0156 ± 0.0009
  τ=0.975: 1.2677 ± 0.0010

--- Avg Shift-RMSE (multiply 10%) ---
  τ=0.025: 2.9384 ± 0.0109
  τ=0.050: 2.7910 ± 0.0111
  τ=0.100: 2.4968 ± 0.0115
  τ=0.500: 0.4097 ± 0.0332
  τ=0.900: 2.3346 ± 0.0017
  τ=0.950: 2.6284 ± 0.0015
  τ=0.975: 2.7756 ± 0.0015
